# Final states of a padded batch

A factory logs three sensor readings at every step of each machine run, and a classifier
reads a whole run at a time. Runs last between 3 and 9 steps, so a batch pads every run
to 9 steps. You will run a recurrent layer over the padded batch in NumPy, take each run's
state at its own last step, and add the right-to-left pass of a bidirectional layer, which
is where padding is easiest to get wrong.

**How to work through it.** Two parts, one function each. The setup cell hands you
`rnn_states`, the lesson's loop in NumPy, and the parts build on it. Replace the `TODO` and
the `raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It also hands you `rnn_states`, the lesson's five-line loop in NumPy:
one step of the whole batch at a time, all `T` steps of every row, no knowledge of padding.
The parts below build on it.

In [ ]:
import numpy as np

# Six runs padded to 9 steps. lengths[i] is the number of real steps of run i.
rng = np.random.default_rng(4)
B, T, D, H = 6, 9, 3, 4
lengths = np.array([9, 4, 7, 3, 9, 5])
X = rng.normal(size=(B, T, D))
X[np.arange(T)[None, :] >= lengths[:, None]] = 0.0     # zero padding after each run ends

# One set of weights per direction, laid out as in nn.RNN: W_xh is (H, D), W_hh is (H, H).
W_xh, W_hh, b = 0.7 * rng.normal(size=(H, D)), 0.8 * rng.normal(size=(H, H)), 0.1 * rng.normal(size=H)
W_xh_r, W_hh_r, b_r = 0.7 * rng.normal(size=(H, D)), 0.8 * rng.normal(size=(H, H)), 0.1 * rng.normal(size=H)
print(f"{B} runs padded to {T} steps, lengths {lengths.tolist()}")

# Given code, not a task: the lesson's five-line loop, the same recurrence in NumPy with the
# batch as rows, one step of the whole batch at a time. It runs all T steps of every row and
# knows nothing about padding; the parts below build on it.
def rnn_states(X, W_xh, W_hh, b):
    """Hidden states after every step, shape (B, T, H), starting from h_0 = 0."""
    B, T, _ = X.shape
    h = np.zeros((B, W_hh.shape[0]))
    states = np.empty((B, T, W_hh.shape[0]))
    for step in range(T):
        h = np.tanh(X[:, step] @ W_xh.T + h @ W_hh.T + b)
        states[:, step] = h
    return states

## Part 1 (coding task): The state at each sequence's own end

Write `final_states(X, lengths, W_xh, W_hh, b)`. It returns an array of shape `(B, H)` whose
row `i` is the hidden state of sequence `i` after its last real step, step `lengths[i]`. This
is the vector a classifier of whole runs reads, and the `h_n` that the PyTorch layers return
for sequences packed with `pack_padded_sequence`.

Do not rely on the padding being zeros. Some pipelines pad with other values, so use
`lengths` to tell real steps from padding.

In [ ]:
def final_states(X, lengths, W_xh, W_hh, b):
    """State of each sequence after its last real step, shape (B, H)."""
    lengths = np.asarray(lengths)
    # TODO
    raise NotImplementedError

In [ ]:
# A small batch built from fixed formulas: 2 sequences, 5 steps, 3 features, 4 hidden units.
Xc = np.sin(np.arange(2 * 5 * 3).reshape(2, 5, 3) * 0.7)
Xc[1, 2:] = 0.0                                   # the second sequence has 2 real steps
Lc = np.array([5, 2])
W1 = 0.8 * np.cos(np.arange(12).reshape(4, 3) * 0.9)
W2 = 0.6 * np.sin(np.arange(16).reshape(4, 4) * 1.3 + 0.5)
b1 = np.linspace(-0.2, 0.3, 4)

F = final_states(Xc, Lc, W1, W2, b1)
want = np.array([[0.55922178, -0.36184352, 0.81214099, 0.17846440],
                 [-0.28976127, -0.57040642, 0.90506911, -0.56870874]])
assert np.shape(F) == (2, 4), f"expected one state per sequence, shape (2, 4), got {np.shape(F)}"
assert np.allclose(F[0], want[0], atol=1e-6), "wrong for the sequence that uses all 5 steps: its final state is the one after step 5"
assert np.allclose(F[1], want[1], atol=1e-6), ("right for the full-length sequence, wrong for the one with 2 real steps: "
                                               "its final state is the one after step 2, before any padding is read")
Xg = Xc.copy()
Xg[1, 2:] = 5.0
assert np.allclose(final_states(Xg, Lc, W1, W2, b1), want, atol=1e-6), \
    "the result changed when the padding was 5.0 instead of 0.0: use lengths, not the padding values"
print("Part 1 passed")

## Part 2 (coding task): The right-to-left pass

A bidirectional layer also reads each run from its end back to its start, with weights of its
own. Write `reverse_final_states(X, lengths, W_xh, W_hh, b)`. For each sequence `i` it returns
the state of this right-to-left pass after it has read the real steps from step `lengths[i]`
back to step 1, so shape `(B, H)`. Again the padding values must not matter.

In [ ]:
def reverse_final_states(X, lengths, W_xh, W_hh, b):
    """State after reading each sequence's real steps from last to first, shape (B, H)."""
    lengths = np.asarray(lengths)
    # TODO
    raise NotImplementedError

In [ ]:
R = reverse_final_states(Xc, Lc, W1, W2, b1)
want = np.array([[-0.42178258, -0.56003273, 0.91317715, -0.46808232],
                 [-0.83662678, 0.93509936, -0.83247279, 0.90965529]])
assert np.shape(R) == (2, 4), f"expected shape (2, 4), got {np.shape(R)}"
flipped = rnn_states(Xc[:, ::-1], W1, W2, b1)[:, -1]
if np.allclose(R[1], flipped[1], atol=1e-6) and not np.allclose(R[1], want[1], atol=1e-6):
    raise AssertionError("for the sequence with 2 real steps this reads the padding first: "
                         "reversing the whole time axis puts its 3 padding steps at the start")
assert np.allclose(R[0], want[0], atol=1e-6), "wrong for the sequence that uses all 5 steps: read it from step 5 back to step 1"
assert np.allclose(R[1], want[1], atol=1e-6), "wrong for the sequence with 2 real steps: read step 2, then step 1, and nothing else"
Xg = Xc.copy()
Xg[1, 2:] = 5.0
assert np.allclose(reverse_final_states(Xg, Lc, W1, W2, b1), want, atol=1e-6), \
    "the result changed when the padding was 5.0 instead of 0.0: use lengths, not the padding values"

# The two passes joined, as a bidirectional layer's summary of each run.
summary = np.concatenate([final_states(X, lengths, W_xh, W_hh, b),
                          reverse_final_states(X, lengths, W_xh_r, W_hh_r, b_r)], axis=1)
naive = rnn_states(X[:, ::-1], W_xh_r, W_hh_r, b_r)[:, -1]
print("summary shape:", summary.shape)
print("lengths:                       ", lengths.tolist())
print("error of reversing the padding:", np.round(np.abs(naive - summary[:, H:]).max(axis=1), 3).tolist())
print("Part 2 passed")

## Solution

**Why it works.** Each row of `X[:, t]` is one sequence's input at step $t$. With
sequences as rows, $W_{xh}\mathbf{x}_t$ for every row at once is `X[:, t] @ W_xh.T`, and the same holds for
the state, so one step of the whole batch is two matrix products. The loop over time stays,
because each state needs the one before it. That loop is the one the lesson prints as five
lines, so it is given here instead of asked for. The padding is what the lesson leaves to
`pack_padded_sequence`, and that is the job of the two parts below.

### Part 1: The state at each sequence's own end

In [ ]:
def final_states(X, lengths, W_xh, W_hh, b):
    """State of each sequence after its last real step, shape (B, H)."""
    lengths = np.asarray(lengths)
    states = rnn_states(X, W_xh, W_hh, b)
    return states[np.arange(len(lengths)), lengths - 1]

**Why it works.** A padded step is still a step of the recurrence. With a zero input the state
keeps moving, since $\tanh(W_{hh}\mathbf{h} + \mathbf{b}) \ne \mathbf{h}$ in general, so the state after
the last column is wrong for every run shorter than `T`. Running every row for all `T` steps
and taking column `lengths[i] - 1` of row `i` with one fancy index gives each run's state
before it has read any padding, whatever the padding holds. The padded steps cost compute
and change nothing that is kept. `pack_padded_sequence` does the same job in PyTorch and
also skips the padded work.

### Part 2: The right-to-left pass

In [ ]:
def reverse_final_states(X, lengths, W_xh, W_hh, b):
    """State after reading each sequence's real steps from last to first, shape (B, H)."""
    lengths = np.asarray(lengths)
    B, T, _ = X.shape
    t = np.arange(T)[None, :]
    # Row i reads steps lengths[i]-1, ..., 0, then its padding where it was.
    src = np.where(t < lengths[:, None], lengths[:, None] - 1 - t, t)
    return final_states(X[np.arange(B)[:, None], src], lengths, W_xh, W_hh, b)

**Why it works.** Reversing the whole time axis puts a short run's padding first, so the
right-to-left pass starts from a state the padding has already moved. The fix reverses only the
real steps: for `t < lengths[i]` the index `lengths[i] - 1 - t` reads run `i` backward, and the
padding stays at the end, where `final_states` never reads it.

On the setup batch the naive reversal is exact for the two runs that fill all 9 steps. Its error
is largest on the 3-step run and smallest on the 7-step run: the state the padding left behind
fades over the real steps that follow, the same fading the lesson measures for gradients. So the
bug is nearly invisible on long runs and large on short ones. PyTorch's bidirectional layers
reverse each packed sequence within its own length, which is one more reason to pack.